In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import os
import requests
from minsearch import Index

In [2]:
openai_client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

In [3]:
def llm(prompt):
    response = openai_client.responses.create(
        model="poolside/laguna-m.1:free",
        input=prompt
    )
    return response.output_text

In [4]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)


Yes, you can typically join a course at any time, depending on the platform or institution offering it. However, the exact details will vary based on the course's structure and policies. Here's what you should do:

1. **Check the Course Details**: Look for information on enrollment deadlines, start dates, and late registration options. If it's a self-paced course, you can usually join immediately. For live or cohort-based courses, there may be specific start dates or deadlines.

2. **Review Requirements**: Ensure you meet any prerequisites (e.g., prior knowledge, software setup) or registration requirements (e.g., payment, account creation).

3. **Catch-Up Options**: If the course is already in progress, check if there are provisions for latecomers, such as extended deadlines or access to past materials.

4. **Contact Support**: If unsure, reach out to the course instructor, administrator, or platform support team for clarification.

Let me know the specific course name or platform, a

In [5]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.
search_results = search(question)
Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [6]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [7]:
answer = llm(prompt)
print(answer)


Yes, you can join the course now. If you want to receive a certificate, ensure you submit your project while submissions are still open. You don’t need to register to participate, but you can submit homework (if the form is still open) without registration. Registration is optional and mainly used to gauge interest before the start date. For participation in "Office Hours" or live sessions, note that the Zoom link is restricted to instructors/presenters/TAs, while students join via YouTube Live and submit questions to Slido.



In [8]:
def rag(question):
     search_results = search(question)
     user_prompt = build_prompt(question, search_results)
     return llm(user_prompt)

In [9]:
docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [10]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1346

In [11]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [12]:
index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [13]:
index.search("I just discovered the course. Can I still join?",filter_dict={"course": "llm-zoomcamp"},num_results=5)

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '9f689c185f',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I missed the first homework - can I still get a certificate?',
  'answer': 'Yes, you need to pass the Cap

In [14]:
def search(question,course="llm-zoomcamp"):
    boost_dict={"question": 2, "section": 0.5}
    filter_dict={"course": course}
    
    return index.search(question, 
                        boost_dict=boost_dict,
                        filter_dict=filter_dict, 
                        num_results=5)


In [15]:
search_results = search(question)

In [16]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

In [17]:
USER_PROMPT_TEMPALATE = '''
Question:
{question}

Context:
{context}
'''

In [18]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc['section'])
        lines.append('Q: ' + doc['question'])
        lines.append('A: ' + doc['answer'])
        lines.append('')

    return '\n'.join(lines).strip()

In [19]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPALATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [20]:
prompt = build_prompt(question, search_results)
print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [21]:
response = openai_client.responses.create(model="poolside/laguna-m.1:free",
                                          input=prompt )

In [22]:
response.output_text

'\n**Answer:**  \nYes, you can join the course now! However, if you want to receive a certificate, you must submit your project by the deadline while submissions are still open. Here’s what you need to know:  \n\n- **Start Immediately**: You don’t need prior registration to begin the course. You can access the videos, GitHub materials, and homework instructions anytime via the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), [logistics guide](https://datatalks.club/docs/courses/zoomcamp-logistics/), and [GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp). The deadlines for assignments are listed on the [course platform](https://courses.datatalks.club/llm-zoomcamp-2026/).  \n\n- **Live Cohort Requirement for Certificates**: To earn a certificate, you must participate in the "live" cohort (not self-paced mode). During the live run, you’ll submit homework, complete a capstone project, and **peer-review 3 other projects**—this step is mandatory for cer

In [23]:
response.output[0].content[0].text

"\nOkay, let's see. The user is asking if they can join the course now that they've just discovered it. I need to check the provided context to answer this.\n\nLooking at the first Q&A, the answer says yes, but if they want a certificate, they have to submit the project while submissions are still open. Then the second Q&A mentions that registration isn't necessary because you're automatically accepted and can start learning and submitting homework even if you haven't registered, as long as the form is open. But the third Q&A states that you can't get a certificate in self-paced mode because peer reviews are needed, and those only happen during the live cohort. \n\nThe fourth answer outlines the workflow, which is to follow the docs and GitHub repo, start anytime, with deadlines on the course platform. Homework submissions are through there too. The fifth Q&A says the next course is Summer 2027, which might be relevant if the current one is ending.\n\nSo putting it all together: The us

In [25]:
print(response.usage)

ResponseUsage(input_tokens=500, input_tokens_details=InputTokensDetails(cached_tokens=5), output_tokens=691, output_tokens_details=OutputTokensDetails(reasoning_tokens=323), total_tokens=1191, cost=0, is_byok=False, cost_details={'upstream_inference_cost': 0, 'upstream_inference_input_cost': 0, 'upstream_inference_output_cost': 0})


In [28]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = response.usage.input_tokens * input_price +response.usage.output_tokens * output_price


cost

0.0034845

In [33]:
def llm(instructions, user_prompt, model="poolside/laguna-m.1:free"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [34]:
def rag(query, model="poolside/laguna-m.1:free"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [35]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)


Yes, you can join the course now. However, if you want to receive a certificate, you must submit your project while the course is still accepting submissions. Registration is not mandatory, and you can start learning and submitting homework as long as the submission form is open. You can begin with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp). Check the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/) for deadlines. Note that certificates are only awarded for live cohort participation, not self-paced mode, due to the requirement for peer review of capstone projects.



In [1]:
rag("How do I get a certificate?")

NameError: name 'rag' is not defined